# Reproducing Grosmark & Buzsáki (2016) from dandiset 000044

Paper: A. D. Grosmark and G. Buzsáki, "Diversity in neural firing dynamics supports both rigid and
learned hippocampal sequences", *Science* 351:1440 (2016),
[doi:10.1126/science.aad1935](https://doi.org/10.1126/science.aad1935).

Dandiset: [DANDI:000044](https://dandiarchive.org/dandiset/000044/0.261005.2101), version
0.261005.2101. The paper cites no version (it predates the dandiset and points to CRCNS). This
is the most recent published version. It is also the one that matches the files best: in the two
earlier versions (0.210812.1516 and 0.250624.0426) the `rate` of every position series holds the
sampling period (0.0256 s) in place of the rate (39.06 Hz), so the positions would span 880 to
2360 hours in place of the MAZE epoch. The cell below shows this. Otherwise the versions differ only
in subject metadata.

Text used: the full main text and the legends of Figures 1 to 4, from the NIH author manuscript in
PubMed Central (PMC4919122). The supplementary Materials and Methods and figs. S1 to S22 could not
be retrieved (PMC and Science serve them only behind a browser check), so every definition the
main text does not give is a common choice, fixed in the claims file before the analysis. Neither
the paper nor the dandiset links analysis code.

Eight claims rest on the selection of strongly contributing cells (PCC). The test fixed for it
before the analysis has a false-positive rate of 56%, far from the 5% that "significant" implies,
so these claims are blocked as `methods_underspecified`; their analyses are still shown, with a
better calibrated variant of the contribution. fig2d-shuffle-high-dpcc ranks all place cells by
dPCC and does not use that selection, so it is tested.

| Claim | Status |
|---|---|
| sessions-and-tracks: 4 rats, 8 sessions, two linear and one circular track | tested |
| place-cell-count: 491 place cells | tested |
| fig1c-post-gt-pre: sequence scores higher in POST than PRE | tested |
| fig1c-maze-gt-pre: sequence scores higher in MAZE than PRE | tested |
| fig2b-strong-contributors: 216 strongly contributing neurons | blocked, methods_underspecified (analysis shown) |
| fig2-pre-post-overlap: 73% of PRE contributors also contribute in POST | blocked, methods_underspecified (analysis shown) |
| fig2d-shuffle-high-dpcc: shuffling the top 10 to 20% dPCC cells removes replay | tested |
| fig2e-remove-plastic: removing plastic cells removes replay | blocked, methods_underspecified (analysis shown) |
| figs2-track-distribution: rigid and plastic cells similarly placed on the track | blocked, methods_underspecified (analysis shown) |
| fig3a-plastic-specificity: plastic cells more specific, fewer fields | blocked, methods_underspecified (analysis shown) |
| fig3b-plastic-improve: plastic cells sharpen during the session | blocked, methods_underspecified (analysis shown) |
| fig4a-rigid-faster: rigid cells fire faster | blocked, methods_underspecified (analysis shown) |
| fig4a-plastic-ripple-burst-coactivation: plastic cells more ripple-related, bursty and coactive | blocked, methods_underspecified (analysis shown) |
| fig4c-pre-fast-coactivation: PRE coactivation dominated by fast cells | tested |
| fig4c-slow-coactivation-increase: slow cells gain coactivation PRE to POST | tested |
| fig4c-slow-replay: slow cells show most replay of coactivation | tested |
| rate-split-medians: low and high-rate medians 0.39 and 1.12 Hz | tested |
| figs16-low-rate-info-gain: low-rate cells more informative, larger ripple-gain increase | tested |
| fig4d-slow-sequences: slow but not fast cells increase sequence content | tested |
| fig4e-gain-sequences: increase limited to high ripple-gain cells | tested |

In [ ]:
import os
for k in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ[k] = "1"
import warnings
warnings.filterwarnings("ignore", category=UserWarning)
from concurrent.futures import ProcessPoolExecutor

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

import grosmark as g

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
print("dandiset", g.DANDISET, "version", g.VERSION)

### Versions

The rate of the position series in the first published version and in the pinned one, against
the length of the MAZE epoch.

In [ ]:
OLD = dict(g.asset_urls("0.210812.1516"))
URLS = g.asset_urls()
rows = []
for path, url in URLS:
    with g.open_nwb(url) as f:
        beh = f["processing/behavior"]
        name = [k for k in beh if "Linearized" in k][0]
        ser = beh[name][list(beh[name])[0]]
        n = ser["data"].shape[0]
        rate_new = float(ser["starting_time"].attrs["rate"])
        ep = f["intervals/epochs"]
        maze = float(ep["stop_time"][1] - ep["start_time"][1])
    with g.open_nwb(OLD[path]) as f:
        beh = f["processing/behavior"]
        ser = beh[name][list(beh[name])[0]]
        rate_old = float(ser["starting_time"].attrs["rate"])
    rows.append(dict(file=path.split("/")[1][:30], samples=n, maze_s=round(maze, 1),
                     rate_0_210812=rate_old, span_old_h=round(n / rate_old / 3600, 1),
                     rate_pinned=round(rate_new, 3), span_pinned_s=round(n / rate_new, 1)))
print(pd.DataFrame(rows).to_string(index=False))

## Data check

Each file is streamed and reduced in parallel: units, epochs, states, positions, one LFP channel
(the good channel with the most 150 to 250 Hz power in a POST non-REM chunk), the ripple envelope
at the candidate events, and the checks below. This takes a few minutes.

In [ ]:
import time
T0 = time.time()
with ProcessPoolExecutor(4) as ex:
    S = list(ex.map(g.prepare_session, *zip(*URLS)))
print(f"loaded in {time.time() - T0:.0f} s")
NAMES = [s["session"] for s in S]
print(len(S), "sessions loaded:", NAMES)

In [ ]:
rows = []
for s in S:
    ep = s["epochs"]
    ct = pd.Series(s["cell_type"]).value_counts()
    loc = pd.Series(s["location"][s["cell_type"] == "excitatory"]).value_counts()
    t, x, v = g.lin_speed(s)
    tx, sp = g.xy_speed(s)
    run = np.isfinite(x) & (np.abs(np.nan_to_num(v)) > g.RUN_SPEED)
    last_spike = max(q.max() for q in s["spikes"] if len(q))
    rows.append(dict(
        subject=s["subject"], session=s["session"], track=s["track"],
        PRE_h=round((ep["PREEpoch"][1] - ep["PREEpoch"][0]) / 3600, 2),
        MAZE_min=round((ep["MazeEpoch"][1] - ep["MazeEpoch"][0]) / 60, 1),
        POST_h=round((ep["POSTEpoch"][1] - ep["POSTEpoch"][0]) / 3600, 2),
        epochs_end_s=round(ep["POSTEpoch"][1], 1), lfp_end_s=round(s["lfp_seconds"], 1),
        last_spike_s=round(last_spike, 1),
        pyr=int(ct.get("excitatory", 0)), int_=int(ct.get("inhibitory", 0)),
        pyr_lCA1=int(loc.get("lCA1", 0)), pyr_rCA1=int(loc.get("rCA1", 0)),
        pos_rate=round(s["lin"]["rate"], 2), pos_unit=s["lin"]["unit"],
        lin_max_m=round(float(np.nanmax(s["lin"]["data"])), 3),
        on_track_frac=round(float(np.isfinite(x).mean()), 3), run_s=round(run.sum() / s["lin"]["rate"], 0),
        lin_speed_run=round(float(np.nanmedian(np.abs(v[run]))), 1),
        xy_speed_run=round(float(np.nanmedian(sp[run])), 1),
        run_up_frac=round(float(np.mean(v[np.isfinite(v)] > g.RUN_SPEED)), 3),
        run_down_frac=round(float(np.mean(v[np.isfinite(v)] < -g.RUN_SPEED)), 3),
        directions=str(g.directions(s)),
    ))
DATA = pd.DataFrame(rows)
print(DATA.to_string(index=False))
print()
print("subjects", DATA.subject.nunique(), "sessions", len(DATA), "pyramidal", DATA.pyr.sum(),
      "interneurons", DATA.int_.sum())
print("sessions per subject:", DATA.groupby("subject").size().to_dict())

In [ ]:
# sleep states, LFP channel and clock checks
rows = []
for s in S:
    st = s["states"]
    dur = pd.Series(st["stop"] - st["start"]).groupby(st["label"]).sum()
    td0, td1 = s["theta_delta"][0.0], s["theta_delta"][600.0]
    ov = s["ripple_overlap"]
    ev = pd.Series(s["events"]["epoch"]).value_counts()
    rows.append(dict(session=s["session"], states=",".join(sorted(set(st["label"]))),
                     awake_h=round(dur.get("Awake", 0) / 3600, 2), nrem_h=round(dur.get("Non-REM", 0) / 3600, 2),
                     rem_h=round(dur.get("REM", 0) / 3600, 2),
                     states_end_s=round(float(st["stop"].max()), 0),
                     channel=s["ripple_channel"], bad_ch=int(s["bad_electrode"].sum()),
                     TD_REM=round(td0["REM"], 2), TD_NREM=round(td0["Non-REM"], 2),
                     TD_REM_shift600=round(td1["REM"], 2), TD_NREM_shift600=round(td1["Non-REM"], 2),
                     burst_rip=round(ov[0.0], 3), burst_rip_shift05=round(ov[0.5], 3),
                     burst_rip_shift2=round(ov[2.0], 3),
                     ev_PRE=int(ev.get("PRE", 0)), ev_MAZE=int(ev.get("MAZE", 0)), ev_POST=int(ev.get("POST", 0))))
print(pd.DataFrame(rows).to_string(index=False))

In [ ]:
# spikes against positions: median error (cm) of decoding position during running, rate maps from
# even minutes and 0.5 s windows from odd minutes, with the spikes shifted by each lag (s)
rows = []
for s in S:
    e = g.position_error(s)
    rows.append(dict(session=s["session"], **{f"lag {k:+g}": round(v, 1) for k, v in e.items()}))
POSERR = pd.DataFrame(rows)
print(POSERR.to_string(index=False))
print("lag with the smallest error per session:",
      POSERR.set_index("session").astype(float).idxmin(axis=1).to_dict())

# recordings deposited twice: identical spike trains or positions across files
sig_units = {}
for s in S:
    for u, q in zip(s["unit_id"], s["spikes"]):
        sig_units.setdefault((len(q), round(float(q[:50].sum()), 5)), []).append((s["session"], int(u)))
dup_units = [v for v in sig_units.values() if len({x[0] for x in v}) > 1]
sig_pos = {}
for s in S:
    sig_pos.setdefault(round(float(np.nansum(s["xy"]["data"])), 4), []).append(s["session"])
print("unit spike trains found in more than one file:", len(dup_units))
print("position series found in more than one file:", [v for v in sig_pos.values() if len(v) > 1])

Checks of the description against the data, from the outputs above:

- Four subjects and eight sessions, each with PRE, MAZE and POST epochs, as the metadata state.
  Durations are printed; the description's "~4 hour" PRE and POST and "~45 minute" MAZE are
  compared there.
- Units are labelled `excitatory` or `inhibitory`; pyramidal cells here are the excitatory units.
  Their hemisphere is in `location` (lCA1 or rCA1); the electrodes' `location` is "unknown".
- Positions are stated in metres. The linearized position spans 0 to 1.6 m on the 1.6 m tracks and
  0 to 2 m on the 2 m track, consistent with the description. On the circular track (described as
  1 m in diameter, a circumference of 3.14 m) it reaches 2.84 to 2.9 m, consistent with the series
  description, which has it run from one edge of the reward area to the other. The running speed
  from the 2D positions agrees with that from the linearized position, so the 2D positions are
  consistent with metres.
- The sleep states are consistent with the LFP: the theta/delta ratio is far higher in REM than in
  non-REM in every session, and the contrast is lost when the labels are shifted by 600 s.
- Spikes and LFP share a clock: the population bursts in non-REM hold a ripple far more often with
  the envelope as stored than shifted by 0.5 or 2 s.
- Spikes and positions share a clock: the decoding error of position is smallest at lag 0.
- No unit spike train or position series appears in more than one file.
- In Gatsby_08022013 the POST epoch ends at 30413.6 s, while the LFP, the spikes and the sleep
  states run to about 32000 s. Firing rates over the session use the LFP duration.
- On the circular track the linearized position mostly decreases while the animal runs
  (`run_down_frac` against `run_up_frac`), so its single template is built from running toward
  decreasing position. The description says the position increases clockwise, not which way the
  animals ran.
- The description mentions neck EMG and accelerometer signals; the files hold neither, and no raw
  wide-band data (`acquisition` is empty). These are not needed here.

## Templates, event scores and per-cell contributions

For each session: rate maps on running in each direction, place cells, the event scores (rZ) of
every candidate event, each active cell's contribution to its events (for PCC), and the same
contributions with every rate map shifted (the false-positive rate of the PCC test). Seeds are
fixed per session and per step. The contributions are computed twice: as the criterion defines
them, on the template the event was scored with, and as a variant on the better of the event's
templates (see fig2b-strong-contributors).

In [ ]:
T0 = time.time()
with ProcessPoolExecutor(4) as ex:
    FP = list(ex.map(g.first_pass, S, [1000 + i for i in range(8)], [2000 + i for i in range(8)],
                     [3000 + i for i in range(8)]))
print(f"first pass in {time.time() - T0:.0f} s")
EVENTS = pd.concat([pd.DataFrame(dict(session=s["session"], subject=s["subject"], epoch=fp["decoded"]["epoch"],
                                      rZ=fp["decoded"]["rZ"], r=fp["decoded"]["r"],
                                      n_active=fp["decoded"]["n_active"]))
                    for s, fp in zip(S, FP)], ignore_index=True)
print("candidate events scored, by session and epoch:")
print(EVENTS.groupby(["session", "epoch"]).size().unstack().to_string())
cand = pd.DataFrame([pd.Series(s["events"]["epoch"]).value_counts() for s in S], index=NAMES)
print("candidate events before the active-cell condition:")
print(cand.to_string())

## sessions-and-tracks

> Recordings were made in four rats, in eight novel exploration sessions, each a PRE sleep, a MAZE
> epoch on one of two linear tracks or a circular track, and a POST sleep.

In [ ]:
order_ok = [s["epochs"]["PREEpoch"][1] <= s["epochs"]["MazeEpoch"][0] + 1e-6 and
            s["epochs"]["MazeEpoch"][1] <= s["epochs"]["POSTEpoch"][0] + 1e-6 for s in S]
print(DATA[["subject", "session", "track", "PRE_h", "MAZE_min", "POST_h"]].to_string(index=False))
print("subjects:", DATA.subject.nunique(), " sessions:", len(DATA), " epochs in PRE, MAZE, POST order:",
      sum(order_ok), "of", len(S))
print("tracks:", DATA.track.value_counts().to_dict())

**Result: reproduced.** The dandiset holds 4 subjects and 8 sessions, each with PRE, MAZE and
POST epochs in that order (8 of 8), on a 1.6 m linear track (4 sessions), a 2 m linear track (1)
and a circular track (3). The rooms named in the paper are not in the files.

## place-cell-count

> The Bayesian decoder was built from 491 place cells over the eight sessions.

A place cell is a pyramidal cell whose smoothed rate map on running (above 5 cm/s) peaks at 1 Hz
or more in at least one direction. 0.5 and 2 Hz are shown as variants.

In [ ]:
rows = []
for s, fp in zip(S, FP):
    pk = fp["tpl"]["peak"]
    rows.append(dict(subject=s["subject"], session=s["session"], pyramidal=len(pk),
                     place_05Hz=int((pk >= 0.5).sum()), place_1Hz=int((pk >= 1).sum()),
                     place_2Hz=int((pk >= 2).sum()),
                     **{f"template_{d:+d}": len(c) for d, c in fp["cells"].items()}))
PC = pd.DataFrame(rows)
print(PC.to_string(index=False))
print("totals:", PC[["pyramidal", "place_05Hz", "place_1Hz", "place_2Hz"]].sum().to_dict())
print("by subject (1 Hz):", PC.groupby("subject").place_1Hz.sum().to_dict())
print("expected 491, tolerance 49; within:", abs(PC.place_1Hz.sum() - 491) <= 49)

**Result: not reproduced.** With a 1 Hz peak, 417 of the 562 pyramidal cells are place cells,
below the paper's 491 by more than the tolerance of 49. With a 0.5 Hz peak the count is 480,
within the tolerance, and with 2 Hz it is 352. The criterion fixes 1 Hz, and the other thresholds
are variants that do not decide the outcome; the count depends strongly on the threshold. The
paper's definition is in the Methods, which were not available.

## fig1c-post-gt-pre and fig1c-maze-gt-pre

> Sequence scores (rZ) of candidate events are higher in POST sleep than in PRE sleep (replay).
> Sequence content is higher in MAZE immobility events than in PRE sleep events.

Kruskal-Wallis across the three epochs on all events pooled, then Tukey's HSD on the ranks.

In [ ]:
def ranks_tukey(groups):
    allv = np.concatenate(groups)
    rk = stats.rankdata(allv)
    out, i = [], 0
    for gr in groups:
        out.append(rk[i:i + len(gr)])
        i += len(gr)
    return stats.tukey_hsd(*out)

ep_names = ["PRE", "MAZE", "POST"]
grp = [EVENTS.loc[EVENTS.epoch == e, "rZ"].dropna().values for e in ep_names]
for e, v in zip(ep_names, grp):
    print(f"{e:5s} n={len(v):5d} mean rZ={v.mean():.3f} SE={v.std(ddof=1) / np.sqrt(len(v)):.3f} "
          f"median={np.median(v):.3f} range {v.min():.2f} to {v.max():.2f}; sign-rank vs 0 p={stats.wilcoxon(v).pvalue:.2g}")
kw = stats.kruskal(*grp)
print(f"Kruskal-Wallis H={kw.statistic:.1f} p={kw.pvalue:.2g}")
tk = ranks_tukey(grp)
for i, j in [(2, 0), (1, 0), (2, 1)]:
    print(f"Tukey on ranks {ep_names[i]} vs {ep_names[j]}: p={tk.pvalue[i, j]:.3g}, "
          f"mean difference {grp[i].mean() - grp[j].mean():+.3f}")
per = EVENTS.groupby(["subject", "session", "epoch"]).rZ.mean().unstack()[ep_names]
per["POST>PRE"] = per.POST > per.PRE
per["MAZE>PRE"] = per.MAZE > per.PRE
print(per.round(3).to_string())
print("sessions POST>PRE:", int(per["POST>PRE"].sum()), "of 8; MAZE>PRE:", int(per["MAZE>PRE"].sum()), "of 8")
print("by subject:")
print(EVENTS.groupby(["subject", "epoch"]).rZ.mean().unstack()[ep_names].round(3).to_string())
fig, ax = plt.subplots(figsize=(5, 3.5))
for e, v in zip(ep_names, grp):
    ax.plot(np.sort(v), np.linspace(0, 1, len(v)), label=e)
ax.set_xlabel("rZ"); ax.set_ylabel("cumulative fraction"); ax.legend()
ax.set_xlim(-3, 8)
print("points outside the x axis:", int(sum(((v < -3) | (v > 8)).sum() for v in grp)))

The weighted correlation threshold is not used here; as a variant, the fraction of events with
rZ above 1.96 per epoch.

In [ ]:
print(EVENTS.assign(sig=EVENTS.rZ > 1.96).groupby("epoch").sig.mean().round(3).to_dict())

Variant, within sessions: the pooled test mixes sessions whose rZ differ in level (the circular
track sessions, with one template, score lower in every epoch) and in the number of events. A
one-sided rank-sum test of POST against PRE within each session, a sign test over the 8 sessions,
and the pooled comparison after subtracting each session's PRE mean.

In [ ]:
rows = []
for (sb, ss), df in EVENTS.groupby(["subject", "session"]):
    a = df.loc[df.epoch == "POST", "rZ"]; b = df.loc[df.epoch == "PRE", "rZ"]; m = df.loc[df.epoch == "MAZE", "rZ"]
    rows.append(dict(subject=sb, session=ss, POST_minus_PRE=a.mean() - b.mean(),
                     p_POST_gt_PRE=stats.mannwhitneyu(a, b, alternative="greater").pvalue,
                     MAZE_minus_PRE=m.mean() - b.mean(),
                     p_MAZE_gt_PRE=stats.mannwhitneyu(m, b, alternative="greater").pvalue))
W = pd.DataFrame(rows)
print(W.round(4).to_string(index=False))
n_pos = int((W.POST_minus_PRE > 0).sum())
print(f"sessions with POST > PRE: {n_pos} of 8, sign test p = {stats.binomtest(n_pos, 8, alternative='greater').pvalue:.4f}; "
      f"with one-sided p < 0.05: {int((W.p_POST_gt_PRE < 0.05).sum())}")
cen = EVENTS.copy()
cen["rZc"] = cen.rZ - cen.groupby("session").rZ.transform(lambda v: v[cen.loc[v.index, "epoch"] == "PRE"].mean())
a = cen.loc[cen.epoch == "POST", "rZc"]; b = cen.loc[cen.epoch == "PRE", "rZc"]
print(f"pooled after subtracting each session's PRE mean: POST {a.mean():+.3f}, rank-sum one-sided p = "
      f"{stats.mannwhitneyu(a, b, alternative='greater').pvalue:.3g}")
print("events per epoch on circular and linear tracks:")
print(EVENTS.merge(DATA[["session", "track"]]).assign(circular=lambda d: d.track == "CircularMaze")
      .groupby(["circular", "epoch"]).rZ.agg(["size", "mean"]).round(3).to_string())

Variant, by track: the criterion's test on the five linear-track sessions alone and on the three
circular-track sessions alone. Then, to compare the event pipeline with the paper's, the number of
scored events per session and epoch and the distribution of MAZE rZ in each session.

In [ ]:
ev_t = EVENTS.merge(DATA[["session", "track"]])
ev_t["linear"] = ~ev_t.track.str.startswith("Circular")
for lin, df in ev_t.groupby("linear"):
    grp_t = [df.loc[df.epoch == e, "rZ"].dropna().values for e in ep_names]
    tk_t = ranks_tukey(grp_t)
    print(("linear" if lin else "circular") + f" tracks ({df.session.nunique()} sessions): Kruskal-Wallis p={stats.kruskal(*grp_t).pvalue:.2g}; "
          + "; ".join(f"{ep_names[i]} {grp_t[i].mean():.3f} vs {ep_names[j]} {grp_t[j].mean():.3f}, Tukey on ranks p={tk_t.pvalue[i, j]:.3g}"
                      for i, j in [(2, 0), (1, 0)]))
print()
print("scored events per session and epoch:")
print(EVENTS.groupby(["session", "epoch"]).size().unstack()[ep_names].to_string())
print("MAZE rZ per session:")
print(EVENTS[EVENTS.epoch == "MAZE"].groupby("session").rZ.describe(percentiles=[0.1, 0.5, 0.9])
      .round(3).to_string())

**Result: fig1c-maze-gt-pre reproduced; fig1c-post-gt-pre not reproduced.** MAZE events score
higher than PRE events (mean rZ 1.270 against 0.770; Tukey on ranks p printed as 0), and MAZE is
higher in all 8 sessions. Pooled over sessions, as the criterion fixes the test, POST does not
exceed PRE (0.763 against 0.770, Tukey on ranks p = 0.412). Within sessions it does: POST is
higher in 8 of 8 sessions (sign test p = 0.0039, one-sided rank-sum p < 0.05 in 6), and with each
session's PRE mean subtracted POST is higher by 0.106 (p = 2.3e-14). The pooled test is
dominated by the circular-track sessions, which score lower in every epoch (PRE 0.161, POST
0.260, against 1.112 and 1.207 on the linear tracks) and contribute 11161 of the 23801 POST
events but 5912 of the 16448 PRE events. The same test run on each kind of track finds POST above
PRE on the five linear-track sessions (Tukey on ranks p = 5.1e-07) and on the three circular-track
sessions (p = 1.1e-05), and MAZE above PRE on both (p printed as 0, and p = 0.0007). The scored
events per session (76 to 953 in MAZE, 1246 to 3294 in PRE, 1136 to 4079 in POST) and the MAZE
rZ of each session are printed for comparison with the paper's figs. S2 to S4, which could not be
read.

## fig2b-strong-contributors

> 216 neurons showed significant PCC scores in PRE or POST and were considered to contribute
> strongly to sequences.

A cell's contribution to an event is |r| minus the mean |r| over 20 decodings with that cell's rate
map circularly shifted, with shifts drawn afresh for every event. Strongly contributing in an
epoch: one-sided Wilcoxon signed-rank test of its contributions against 0, p < 0.05, with at
least 5 events. The false-positive rate comes from the same test with every rate map shifted.

On the linear tracks the criterion takes |r| on the template the event was scored with, the one of
the two with the larger rZ. That |r| was selected for being the larger, so it tends to fall when
any part of the template is perturbed, whatever the cell. A variant ("best template") removes
this: |r| is the larger over the event's templates both with the original maps and with the
cell's map shifted.

In [ ]:
def pcc_table(rows, s, place):
    df = pd.DataFrame([(e, ep, c, r0 - np.nanmean(rs)) for e, ep, c, r0, rs in rows],
                      columns=["event", "epoch", "cell", "contrib"])
    out = []
    for j in place:
        rec = dict(session=s["session"], subject=s["subject"], cell=j)
        for e in ("PRE", "POST"):
            v = df.loc[(df.cell == j) & (df.epoch == e), "contrib"].values
            rec[f"n_{e}"] = len(v)
            rec[f"PCC_{e}"] = v.mean() if len(v) else np.nan
            rec[f"p_{e}"] = stats.wilcoxon(v, alternative="greater").pvalue if len(v) >= 5 and np.any(v != 0) else np.nan
        out.append(rec)
    return pd.DataFrame(out)

def strong_cols(df):
    df["strong_PRE"] = df.p_PRE < 0.05
    df["strong_POST"] = df.p_POST < 0.05
    df["strong"] = df.strong_PRE | df.strong_POST
    df["dPCC"] = df.PCC_POST - df.PCC_PRE
    return df

tables = {}
for key in ("contrib", "contrib_null", "contrib_best", "contrib_null_best"):
    frames = []
    for s, fp in zip(S, FP):
        place = np.flatnonzero(fp["tpl"]["peak"] >= g.PLACE_PEAK)
        frames.append(pcc_table(fp[key], s, place))
    tables[key] = strong_cols(pd.concat(frames, ignore_index=True))
CELLS, NULL = tables["contrib"], tables["contrib_null"]
BEST, NULL_BEST = tables["contrib_best"], tables["contrib_null_best"]
# PCC from the even-numbered candidate events only, for the cross-validated controls of fig2d and fig2e
EVEN = strong_cols(pd.concat([pcc_table([r for r in fp["contrib"] if r[0] % 2 == 0], s,
                                        np.flatnonzero(fp["tpl"]["peak"] >= g.PLACE_PEAK))
                              for s, fp in zip(S, FP)], ignore_index=True))
print("place cells", len(CELLS), "; with >= 5 events in PRE:", int((CELLS.n_PRE >= 5).sum()),
      "in POST:", int((CELLS.n_POST >= 5).sum()))
print("strongly contributing: PRE", int(CELLS.strong_PRE.sum()), " POST", int(CELLS.strong_POST.sum()),
      " either", int(CELLS.strong.sum()), f"({CELLS.strong.mean():.1%} of place cells)")
print("false-positive rate (all maps shifted): PRE", f"{NULL.strong_PRE.mean():.3f}", " POST",
      f"{NULL.strong_POST.mean():.3f}", " either", f"{NULL.strong.mean():.3f}",
      f"-> expected {NULL.strong.mean() * len(CELLS):.0f} cells by chance")
print("expected 216, tolerance 21.6; within:", abs(CELLS.strong.sum() - 216) <= 21.6)
for a in (0.01, 0.001):
    n = int(((CELLS.p_PRE < a) | (CELLS.p_POST < a)).sum())
    nn = ((NULL.p_PRE < a) | (NULL.p_POST < a)).mean()
    print(f"variant alpha={a}: {n} strongly contributing, false-positive rate {nn:.3f}")
print(CELLS.groupby(["subject", "session"]).agg(place=("cell", "size"), strong=("strong", "sum"),
                                                strong_PRE=("strong_PRE", "sum"), strong_POST=("strong_POST", "sum"))
      .join(BEST.groupby(["subject", "session"]).agg(strong_best=("strong", "sum")))
      .join(NULL.groupby(["subject", "session"]).agg(fpr=("strong", "mean")))
      .join(NULL_BEST.groupby(["subject", "session"]).agg(fpr_best=("strong", "mean"))).round(3).to_string())
TRACK = DATA.set_index("session").track
print("false-positive rate by track (either epoch): criterion",
      NULL.groupby(NULL.session.map(TRACK)).strong.mean().round(3).to_dict(),
      " best template", NULL_BEST.groupby(NULL_BEST.session.map(TRACK)).strong.mean().round(3).to_dict())
print()
print("variant, best template:")
print("strongly contributing: PRE", int(BEST.strong_PRE.sum()), " POST", int(BEST.strong_POST.sum()),
      " either", int(BEST.strong.sum()), f"({BEST.strong.mean():.1%} of place cells)")
print("false-positive rate (all maps shifted): PRE", f"{NULL_BEST.strong_PRE.mean():.3f}", " POST",
      f"{NULL_BEST.strong_POST.mean():.3f}", " either", f"{NULL_BEST.strong.mean():.3f}",
      f"-> expected {NULL_BEST.strong.mean() * len(BEST):.0f} cells by chance")
for a in (0.01, 0.001):
    n = int(((BEST.p_PRE < a) | (BEST.p_POST < a)).sum())
    nn = ((NULL_BEST.p_PRE < a) | (NULL_BEST.p_POST < a)).mean()
    print(f"best template, alpha={a}: {n} strongly contributing, false-positive rate {nn:.3f}")
print("dPCC of the criterion against the best-template dPCC, Spearman over place cells:",
      round(stats.spearmanr(CELLS.dPCC, BEST.dPCC, nan_policy="omit").statistic, 3))

**Result: blocked (methods_underspecified).** The test fixed in the criterion finds 303 strongly
contributing cells (72.7% of 417 place cells), against the paper's 216, but with every rate map
shifted it calls 56.1% of cells significant (0.843 on the 1.6 m tracks, 0.905 on the 2 m track,
0.140 on the circular track), so about 234 would be expected by chance. The excess is on the
linear tracks, where the event is scored on the template with the larger rZ. The best-template
variant brings the false-positive rate down to 0.206 (0.132 in PRE, 0.118 in POST; 0.158 to 0.262
by track) and finds 188 cells, outside the tolerance; at alpha 0.001 its false-positive rate is
0.031 and it finds 95 cells. Neither the criterion's test nor the variant at alpha 0.05 has the 5%
false-positive rate that a significance test implies, so neither selection can be taken as the
paper's.

## fig2-pre-post-overlap

> Most neurons strongly contributing to PRE sequences (73%) also contributed to POST sequences.

In [ ]:
def overlap(T, label):
    pre = T[T.strong_PRE]
    frac = pre.strong_POST.mean()
    print(f"{label}: PRE strongly contributing {len(pre)}; also in POST {int(pre.strong_POST.sum())} ({frac:.1%}); "
          f"chance (fraction of all place cells strongly contributing in POST) {T.strong_POST.mean():.1%}")
    return frac
frac = overlap(CELLS, "criterion")
tab = pd.crosstab(CELLS.strong_PRE, CELLS.strong_POST)
print(tab.to_string())
print("Fisher exact test of association p =", f"{stats.fisher_exact(tab.values, alternative='greater').pvalue:.2g}")
print("by subject:", CELLS[CELLS.strong_PRE].groupby("subject").strong_POST.agg(["size", "mean"]).round(3).to_dict("index"))
print("expected 73%, tolerance 7.3; within:", abs(frac * 100 - 73) <= 7.3)
overlap(BEST, "variant, best template")

**Result: blocked (methods_underspecified)**, as it depends on fig2b-strong-contributors. Of 248
PRE strongly contributing cells, 235 (94.8%) also contribute in POST, against a chance level of
69.5%. With the best-template variant 72 of 102 (70.6%), within 73 +/- 7.3, against a chance
level of 37.9%.

## Rigid and plastic cells

Strongly contributing cells with a PCC in both PRE and POST, split at the median dPCC of the
session's strongly contributing cells (reading (a)); reading (b), the median over all of the
session's place cells, is a variant. A second variant (`cls_v`) takes the strongly contributing
cells and dPCC of the best-template contributions, split as in reading (a).

In [ ]:
CELLS["cls"] = ""
CELLS["cls_b"] = ""
CELLS["cls_v"] = ""
CELLS["dPCC_best"] = BEST.dPCC.values
CELLS["dPCC_even"] = EVEN.dPCC.values
CELLS["cls_cv"] = ""
for sess, df in CELLS.groupby("session"):
    ok = df.strong & df.dPCC.notna()
    med = df.loc[ok, "dPCC"].median()
    CELLS.loc[ok[ok].index, "cls"] = np.where(df.loc[ok, "dPCC"] < med, "rigid", "plastic")
    med_b = df.dPCC.median()
    CELLS.loc[ok[ok].index, "cls_b"] = np.where(df.loc[ok, "dPCC"] < med_b, "rigid", "plastic")
    medc = df.loc[ok, "dPCC_even"].median()
    CELLS.loc[ok[ok].index, "cls_cv"] = np.where(df.loc[ok, "dPCC_even"] < medc, "rigid", "plastic")
    okv = BEST.loc[df.index, "strong"] & df.dPCC_best.notna()
    medv = df.loc[okv, "dPCC_best"].median()
    CELLS.loc[okv[okv].index, "cls_v"] = np.where(df.loc[okv, "dPCC_best"] < medv, "rigid", "plastic")
print("strong cells without a PCC in one epoch (not classified):", int((CELLS.strong & CELLS.dPCC.isna()).sum()))
print(CELLS[CELLS.cls != ""].groupby(["subject", "session", "cls"]).size().unstack().to_string())
print("totals (a):", CELLS.cls.value_counts().to_dict(), " (b):", CELLS.cls_b.value_counts().to_dict(),
      " best-template variant:", CELLS.cls_v.value_counts().to_dict())
print("cells classed in both (a) and the best-template variant:", int(((CELLS.cls != "") & (CELLS.cls_v != "")).sum()),
      "; same class:", int(((CELLS.cls != "") & (CELLS.cls == CELLS.cls_v)).sum()))
print("dPCC from even events against dPCC from all events, Spearman:",
      round(stats.spearmanr(CELLS.dPCC_even, CELLS.dPCC, nan_policy="omit").statistic, 3))

Per-cell measures: rates, ripple gain, bursting, spatial information, fields, the in-field ratio
over the session, and the 100 ms coactivation matrices.

In [ ]:
MEAS = []
for s, fp in zip(S, FP):
    place = np.flatnonzero(fp["tpl"]["peak"] >= g.PLACE_PEAK)
    MEAS.append(g.cell_measures(s, fp["tpl"], place))
cols = ["rate_session", "rate_sleep", "rate_prepost", "gain", "gain_pre", "gain_post", "burst", "info",
        "n_fields", "peak_pos", "best_dir", "infield_change"]
CELLS = CELLS.join(pd.concat([pd.DataFrame({c: m[c] for c in cols}) for m in MEAS], ignore_index=True))
CELLS["gain_change"] = CELLS.gain_post - CELLS.gain_pre
CELLS["coact"] = np.concatenate([np.nanmean(np.where(np.eye(len(m["C_session"]), dtype=bool), np.nan, m["C_session"]), 1)
                                 for m in MEAS])
print(CELLS[cols + ["gain_change", "coact"]].describe().T[["count", "min", "50%", "max"]].round(3).to_string())

### Re-decoding for the variant templates

Within each session: the top or bottom 10, 20, 50 and 75% of place cells by dPCC shuffled
(fig2d), plastic or rigid cells removed (fig2e), templates of the low or high-rate half (fig4d)
and of the low or high-gain half (fig4e). The rate split is made here with reading (a), a median
split within each session; the section rate-split-medians shows both readings. 50 column-cycle
shuffles per event here, in place of 200, so the `base` scores differ a little from those of
fig1c. The variants `no_plastic_v` and `no_rigid_v` use the best-template classes.

Controls for the selection by dPCC, which is measured on the same PRE and POST events that are
decoded again: `random{X}` shuffles a random X% of the place cells with a dPCC, and `no_random`
removes as many randomly chosen strongly contributing cells as there are plastic cells. A
cross-validated control (`_cv`) ranks and classes cells by the dPCC of the even-numbered
candidate events only, and its replay table uses only the odd-numbered events, which played no
part in choosing the cells. `_best` variants rank cells by the best-template dPCC.

Replay is judged, as the criteria fix it, on all events pooled (`present`); `diff_within` and
`p_within` are a variant with each session's PRE mean subtracted from its PRE and POST scores. The
same table is then given for the five linear-track sessions alone.

In [ ]:
LEVELS = [10, 20, 50, 75]
def variants_for(i, s, fp):
    df = CELLS[CELLS.session == s["session"]]
    cells = fp["cells"]
    rng = np.random.default_rng(9000 + i)
    def restrict(keep):
        keep = set(int(k) for k in keep)
        return {d: np.array([c for c in cl if c in keep], int) for d, cl in cells.items()}
    def drop(rem):
        rem = set(int(k) for k in rem)
        return {d: np.array([c for c in cl if c not in rem], int) for d, cl in cells.items()}
    v = {"base": (cells, None, 5000 + i)}
    ranked = df[df.dPCC.notna()].sort_values("dPCC")
    n = len(ranked)
    for k, X in enumerate(LEVELS):
        m = int(round(n * X / 100))
        v[f"top{X}"] = (cells, ranked.cell.values[n - m:], 5100 + 10 * k + i)
        v[f"bottom{X}"] = (cells, ranked.cell.values[:m], 5200 + 10 * k + i)
    for k, X in enumerate((10, 20, 75)):
        m = int(round(n * X / 100))
        v[f"random{X}"] = (cells, rng.choice(ranked.cell.values, m, replace=False), 5300 + 10 * k + i)
    ranked_e = df[df.dPCC_even.notna()].sort_values("dPCC_even")
    ranked_b = df[df.dPCC_best.notna()].sort_values("dPCC_best")
    for k, (name, X) in enumerate([("top", 10), ("top", 20), ("bottom", 75)]):
        for tag, rk, base_seed in (("cv", ranked_e, 5400), ("best", ranked_b, 5500)):
            m = int(round(len(rk) * X / 100))
            sel = rk.cell.values[len(rk) - m:] if name == "top" else rk.cell.values[:m]
            v[f"{name}{X}_{tag}"] = (cells, sel, base_seed + 10 * k + i)
    v["no_plastic_cv"] = (drop(df.cell[df.cls_cv == "plastic"]), None, 6500 + i)
    v["no_rigid_cv"] = (drop(df.cell[df.cls_cv == "rigid"]), None, 6600 + i)
    v["no_plastic"] = (drop(df.cell[df.cls == "plastic"]), None, 6000 + i)
    v["no_rigid"] = (drop(df.cell[df.cls == "rigid"]), None, 6100 + i)
    classed = df.cell[df.cls != ""].values
    v["no_random"] = (drop(rng.choice(classed, int((df.cls == "plastic").sum()), replace=False)), None, 6400 + i)
    v["no_plastic_v"] = (drop(df.cell[df.cls_v == "plastic"]), None, 6200 + i)
    v["no_rigid_v"] = (drop(df.cell[df.cls_v == "rigid"]), None, 6300 + i)
    rs = df.rate_sleep.rank(method="first").values
    low = rs <= len(df) // 2
    v["low_rate"] = (restrict(df.cell[low]), None, 7000 + i)
    v["high_rate"] = (restrict(df.cell[~low]), None, 7100 + i)
    gr = df.gain.rank(method="first").values
    hi = gr > len(df) // 2
    v["high_gain"] = (restrict(df.cell[hi]), None, 8000 + i)
    v["low_gain"] = (restrict(df.cell[~hi]), None, 8100 + i)
    return v

VARS = [variants_for(i, s, fp) for i, (s, fp) in enumerate(zip(S, FP))]
T0 = time.time()
with ProcessPoolExecutor(4) as ex:
    VRES = list(ex.map(g.variant_pass, S, [fp["tpl"] for fp in FP], VARS))

print(f"variants in {time.time() - T0:.0f} s")
LINEAR = np.array([not s["track"].startswith("Circular") for s in S])

def replay(name, keep=None, odd=False):
    R = [r for r, k in zip(VRES, keep if keep is not None else [True] * len(VRES)) if k]
    if odd:   # only the odd-numbered candidate events
        R = [{name: {q: w[r[name]["index"] % 2 == 1] for q, w in r[name].items()}} for r in R]
    pre = np.concatenate([r[name]["rZ"][r[name]["epoch"] == "PRE"] for r in R])
    post = np.concatenate([r[name]["rZ"][r[name]["epoch"] == "POST"] for r in R])
    p = stats.mannwhitneyu(post, pre, alternative="greater").pvalue
    sess = sum(np.nanmean(r[name]["rZ"][r[name]["epoch"] == "POST"]) > np.nanmean(r[name]["rZ"][r[name]["epoch"] == "PRE"])
               for r in R)
    # variant: each session's PRE mean subtracted from its PRE and POST scores
    pre_c = np.concatenate([r[name]["rZ"][r[name]["epoch"] == "PRE"] - r[name]["rZ"][r[name]["epoch"] == "PRE"].mean() for r in R])
    post_c = np.concatenate([r[name]["rZ"][r[name]["epoch"] == "POST"] - r[name]["rZ"][r[name]["epoch"] == "PRE"].mean() for r in R])
    pc = stats.mannwhitneyu(post_c, pre_c, alternative="greater").pvalue
    return dict(variant=name, n_PRE=len(pre), n_POST=len(post), mean_PRE=pre.mean(), mean_POST=post.mean(),
                diff=post.mean() - pre.mean(), p_one_sided=p, present=p < 0.05, sessions_POST_gt_PRE=sess,
                diff_within=post_c.mean(), p_within=pc)

REPLAY = pd.DataFrame([replay(k) for k in VARS[0]]).set_index("variant")
REPLAY_LIN = pd.DataFrame([replay(k, LINEAR) for k in VARS[0]]).set_index("variant")
CV_NAMES = ["base", "top10_cv", "top20_cv", "bottom75_cv", "no_plastic_cv", "no_rigid_cv"]
REPLAY_CV = pd.DataFrame([replay(k, odd=True) for k in CV_NAMES]).set_index("variant")
REPLAY_CV_LIN = pd.DataFrame([replay(k, LINEAR, odd=True) for k in CV_NAMES]).set_index("variant")
print("all 8 sessions:")
print(REPLAY.round(4).to_string())
print()
print(f"linear-track sessions only ({int(LINEAR.sum())}):")
print(REPLAY_LIN.round(4).to_string())

## fig2d-shuffle-high-dpcc

> The PRE to POST increase in sequence score is eliminated when the place fields of the 10 to 20%
> of cells with the highest dPCC are shuffled, and remains after shuffling the bottom 75%.

In [ ]:
COLS = ["n_PRE", "n_POST", "mean_PRE", "mean_POST", "diff", "p_one_sided", "present", "sessions_POST_gt_PRE", "diff_within", "p_within"]
rows_2d = ["base"] + [f"top{X}" for X in LEVELS] + [f"bottom{X}" for X in LEVELS] + ["random10", "random20", "random75",
                                                                         "top10_best", "top20_best", "bottom75_best"]
print(REPLAY.loc[rows_2d, COLS].round(4).to_string())
elim = (not REPLAY.loc["top10", "present"]) or (not REPLAY.loc["top20", "present"])
remain = REPLAY.loc["bottom75", "present"]
print("replay absent after shuffling top 10% or 20%:", elim, "; present after shuffling bottom 75%:", remain)
print("linear-track sessions only:")
print(REPLAY_LIN.loc[rows_2d, COLS].round(4).to_string())
print("cross-validated: cells ranked on even events, replay on odd events, all sessions:")
print(REPLAY_CV.loc[["base", "top10_cv", "top20_cv", "bottom75_cv"], COLS].round(4).to_string())
print("cross-validated, linear-track sessions only:")
print(REPLAY_CV_LIN.loc[["base", "top10_cv", "top20_cv", "bottom75_cv"], COLS].round(4).to_string())
def by_subject(name):
    out = {}
    for s, r in zip(S, VRES):
        d = out.setdefault(s["subject"], {"PRE": [], "POST": []})
        for e in ("PRE", "POST"):
            d[e].append(r[name]["rZ"][r[name]["epoch"] == e])
    return {k: round(np.concatenate(v["POST"]).mean() - np.concatenate(v["PRE"]).mean(), 3) for k, v in out.items()}
for name in ("base", "top10", "top20", "bottom75", "random10", "random20"):
    print(name, "POST minus PRE mean rZ by subject:", by_subject(name))

**Result: partially reproduced.** The criterion ranks all place cells by dPCC and does not use the
selection of fig2b-strong-contributors, so the claim is tested. On the pooled test of the
criterion replay is absent after shuffling the top 10% (POST minus PRE -0.100, p = 1.0) and the
top 20% (-0.127, p = 1.0), but it is also absent after shuffling the bottom 75% (+0.022,
p = 0.21), so one of the two conditions holds. The pooled test finds no replay before any
shuffling either (-0.008, p = 0.916), so on all sessions it cannot show a removal.

On the five linear-track sessions alone, replay is present before shuffling (+0.101,
p < 0.0001), absent after shuffling the top 10% (+0.015, p = 0.23) or 20% (-0.039, p = 0.995),
and present after shuffling the bottom 75% (+0.098, p < 0.0001), the paper's pattern. Shuffling
a random 10% or 20% of the cells leaves it (+0.099 and +0.088, p < 0.0001); ranking by the
best-template dPCC gives the same pattern. dPCC is measured on the same events that are decoded
again. When cells are ranked on the even-numbered events and replay is measured on the
odd-numbered ones, replay on the linear tracks falls from +0.116 to +0.046 (p = 0.024) after
shuffling the top 10% and to +0.042 (p = 0.038) after the top 20%, and remains after shuffling the
bottom 75% (+0.081, p = 0.0002). Out of sample, shuffling the high-dPCC cells reduces replay but
does not remove it.

## fig2e-remove-plastic

> The PRE to POST increase in sequence content is attributable to a small number of plastic cells;
> removing them, unlike removing rigid cells, removes the increase.

In [ ]:
rows_2e = ["base", "no_plastic", "no_rigid", "no_random", "no_plastic_v", "no_rigid_v"]
print(REPLAY.loc[rows_2e, COLS].round(4).to_string())
print("linear-track sessions only:")
print(REPLAY_LIN.loc[rows_2e, COLS].round(4).to_string())
print("cross-validated: classes from even events, replay on odd events, all sessions:")
print(REPLAY_CV.loc[["base", "no_plastic_cv", "no_rigid_cv"], COLS].round(4).to_string())
print("cross-validated, linear-track sessions only:")
print(REPLAY_CV_LIN.loc[["base", "no_plastic_cv", "no_rigid_cv"], COLS].round(4).to_string())
for name in ("no_plastic", "no_rigid", "no_random"):
    print(name, "POST minus PRE mean rZ by subject:", by_subject(name))
print("ratio of differences (no plastic / no rigid):",
      round(REPLAY.loc["no_plastic", "diff"] / REPLAY.loc["no_rigid", "diff"], 3))

**Result: blocked (methods_underspecified)**, as it depends on fig2b-strong-contributors. On the
pooled test, removing plastic cells gives POST minus PRE -0.168 (p = 1.0) and removing rigid
cells +0.038 (p = 0.019), the pattern the criterion asks for; removing as many randomly chosen
strongly contributing cells gives -0.023 (p = 0.99). On the linear-track sessions alone the three
give -0.060 (p = 0.9997), +0.136 and +0.094 (both p < 0.0001). With classes from the
even-numbered events and replay on the odd-numbered ones, the linear tracks give -0.009
(p = 0.54) without plastic cells and +0.090 (p = 0.0027) without rigid cells. With the
best-template classes, pooled over all sessions, -0.125 (p = 1.0) and +0.020 (p = 0.24).

## figs2-track-distribution

> Rigid and plastic neurons were similarly distributed along the track.

In [ ]:
R = CELLS[CELLS.cls == "rigid"]; P = CELLS[CELLS.cls == "plastic"]
ks = stats.ks_2samp(R.peak_pos, P.peak_pos)
print(f"peak position (fraction of track): rigid n={len(R)} median {R.peak_pos.median():.3f}; "
      f"plastic n={len(P)} median {P.peak_pos.median():.3f}; KS D={ks.statistic:.3f} p={ks.pvalue:.3g}")
Rb = CELLS[CELLS.cls_b == "rigid"]; Pb = CELLS[CELLS.cls_b == "plastic"]
print(f"reading (b): KS p={stats.ks_2samp(Rb.peak_pos, Pb.peak_pos).pvalue:.3g}")
Rv = CELLS[CELLS.cls_v == "rigid"]; Pv = CELLS[CELLS.cls_v == "plastic"]
print(f"best-template variant: rigid n={len(Rv)} plastic n={len(Pv)}, KS D={stats.ks_2samp(Rv.peak_pos, Pv.peak_pos).statistic:.3f} "
      f"p={stats.ks_2samp(Rv.peak_pos, Pv.peak_pos).pvalue:.3g}")
print("by subject (median peak position):")
print(CELLS[CELLS.cls != ""].groupby(["subject", "cls"]).peak_pos.median().unstack().round(3).to_string())

**Result: blocked (methods_underspecified)**, as it depends on fig2b-strong-contributors. The
peak positions of rigid and plastic cells do not differ by the KS test fixed in the criterion
(medians 0.419 and 0.486 of the track, D = 0.145, p = 0.072); with reading (b) p = 0.079 and with
the best-template classes p = 0.12. All three are close to the threshold.

## fig3a-plastic-specificity

> Plastic neurons had higher place-specific indices and fewer place fields than rigid cells.

In [ ]:
def compare(col, a="plastic", b="rigid", cls="cls", alt="greater"):
    A = CELLS.loc[CELLS[cls] == a, col].dropna(); B = CELLS.loc[CELLS[cls] == b, col].dropna()
    p = stats.mannwhitneyu(A, B).pvalue
    print(f"{col}: {a} n={len(A)} median {A.median():.3f} mean {A.mean():.3f}; {b} n={len(B)} median "
          f"{B.median():.3f} mean {B.mean():.3f}; rank-sum p={p:.3g}")
    return A, B, p
_, _, p_info = compare("info")
_, _, p_nf = compare("n_fields")
print("reading (b):"); compare("info", cls="cls_b"); compare("n_fields", cls="cls_b")
print("best-template variant:"); compare("info", cls="cls_v"); compare("n_fields", cls="cls_v")
print(CELLS[CELLS.cls != ""].groupby(["subject", "cls"])[["info", "n_fields"]].mean().unstack().round(3).to_string())

**Result: blocked (methods_underspecified)**, as it depends on fig2b-strong-contributors. Plastic
cells carry more spatial information than rigid cells (median 0.904 against 0.594 bits per
spike, p = 1.9e-05; best-template classes p = 0.0035), but their number of fields does not differ
(means 1.484 and 1.685, p = 0.073; best-template classes p = 0.10).

## fig3b-plastic-improve

> Plastic but not rigid neurons increased their spatial specificity (in-field versus out-of-field
> firing ratio) steadily during the MAZE session.

In [ ]:
for c in ("plastic", "rigid"):
    v = CELLS.loc[CELLS.cls == c, "infield_change"]
    print(f"{c}: n={v.notna().sum()} (no field: {v.isna().sum()}), median change of log10 ratio "
          f"{v.median():.3f}, mean {v.mean():.3f}, cells increasing {(v > 0).sum()}; one-sided sign-rank p="
          f"{stats.wilcoxon(v.dropna(), alternative='greater').pvalue:.3g}")
A = CELLS.loc[CELLS.cls == "plastic", "infield_change"].dropna(); B = CELLS.loc[CELLS.cls == "rigid", "infield_change"].dropna()
print(f"plastic vs rigid rank-sum p={stats.mannwhitneyu(A, B).pvalue:.3g}")
for c in ("plastic", "rigid"):
    v = CELLS.loc[CELLS.cls_b == c, "infield_change"].dropna()
    print(f"reading (b) {c}: n={len(v)} median {v.median():.3f} p={stats.wilcoxon(v, alternative='greater').pvalue:.3g}")
for c in ("plastic", "rigid"):
    v = CELLS.loc[CELLS.cls_v == c, "infield_change"].dropna()
    print(f"best-template variant {c}: n={len(v)} median {v.median():.3f} p={stats.wilcoxon(v, alternative='greater').pvalue:.3g}")
print(CELLS[CELLS.cls != ""].groupby(["subject", "cls"]).infield_change.median().unstack().round(3).to_string())

**Result: blocked (methods_underspecified)**, as it depends on fig2b-strong-contributors. Both
groups increase their in-field ratio over the session (change of log10 ratio, plastic median
0.155, p = 0.0079; rigid 0.129, p = 0.030; between groups p = 0.41). With the best-template classes
plastic cells increase (p = 0.020) and rigid cells do not (p = 0.28).

## fig4a-rigid-faster

> Session-wide firing rates of rigid neurons were higher than those of plastic neurons.

In [ ]:
compare("rate_session", a="rigid", b="plastic")
print("reading (b):"); compare("rate_session", a="rigid", b="plastic", cls="cls_b")
print("best-template variant:"); compare("rate_session", a="rigid", b="plastic", cls="cls_v")
print(CELLS[CELLS.cls != ""].groupby(["subject", "cls"]).rate_session.median().unstack().round(3).to_string())

**Result: blocked (methods_underspecified)**, as it depends on fig2b-strong-contributors. Rigid
cells fire somewhat faster (median 0.700 against 0.591 Hz) without significance (p = 0.062;
reading (b) p = 0.053; best-template classes p = 0.74).

## fig4a-plastic-ripple-burst-coactivation

> Ripple-related spiking, bursting and pairwise coactivation were higher in plastic than in rigid
> neurons.

In [ ]:
res = {}
for col in ("gain", "burst", "coact"):
    A, B, p = compare(col)
    res[col] = (A.median() > B.median()) and p < 0.05
print("higher in plastic with p < 0.05:", res, "->", sum(res.values()), "of 3")
print("reading (b):")
for col in ("gain", "burst", "coact"):
    compare(col, cls="cls_b")
print("best-template variant:")
for col in ("gain", "burst", "coact"):
    compare(col, cls="cls_v")
print(CELLS[CELLS.cls != ""].groupby(["subject", "cls"])[["gain", "burst", "coact"]].median().unstack().round(3).to_string())

**Result: blocked (methods_underspecified)**, as it depends on fig2b-strong-contributors. Ripple
gain (4.489 against 3.939, p = 0.015) and bursting (0.180 against 0.147, p = 0.028) are higher in
plastic cells; coactivation is not (0.028 against 0.027, p = 0.90). With the best-template
classes none reaches p < 0.05 (gain p = 0.15, bursting p = 0.11, coactivation p = 0.41).

## fig4c: coactivation by firing rate

Place cells of each session sorted by session firing rate; windows of 20% of the cells stepped
by 1%. For each window: mean pairwise correlation (100 ms bins) in PRE, the mean POST minus PRE
change, and the partial correlation across its pairs of MAZE running and POST coactivation given
PRE. Curves are averaged over sessions by window position.

In [ ]:
POS = np.arange(0, 81)
curves = {"pre": [], "change": [], "replay": []}
for s, fp, m in zip(S, FP, MEAS):
    df = CELLS[CELLS.session == s["session"]].reset_index(drop=True)
    order = np.argsort(df.rate_session.values)
    N = len(order); w = max(2, int(round(0.2 * N)))
    cp, cc, cr = [], [], []
    for p in POS:
        a = int(round(p / 100 * N)); idx = order[a:a + w]
        iu = np.triu_indices(len(idx), 1)
        pre_ = m["C_pre"][np.ix_(idx, idx)][iu]; post_ = m["C_post"][np.ix_(idx, idx)][iu]; run_ = m["C_run"][np.ix_(idx, idx)][iu]
        cp.append(np.nanmean(pre_)); cc.append(np.nanmean(post_ - pre_)); cr.append(g.partial_corr(run_, post_, pre_))
    curves["pre"].append(cp); curves["change"].append(cc); curves["replay"].append(cr)
CURVES = {k: np.array(v) for k, v in curves.items()}
COACT = {}
for k, v in CURVES.items():
    mean = np.nanmean(v, 0)
    rho = stats.spearmanr(POS, mean)
    per = [stats.spearmanr(POS, row, nan_policy="omit").statistic for row in v]
    COACT[k] = (rho.statistic, per)
    print(f"{k}: averaged curve slowest window {mean[0]:.4f}, fastest window {mean[-1]:.4f}, range {np.nanmin(mean):.4f} "
          f"to {np.nanmax(mean):.4f}; Spearman rho with window position {rho.statistic:.3f} (p={rho.pvalue:.2g}); "
          f"per session rho {np.round(per, 2).tolist()}; positive in {sum(r > 0 for r in per)} of 8")
fig, axs = plt.subplots(1, 3, figsize=(11, 3))
for ax, k in zip(axs, CURVES):
    ax.plot(POS, CURVES[k].T, color="0.8", lw=0.8); ax.plot(POS, np.nanmean(CURVES[k], 0), color="k")
    ax.set_title(k); ax.set_xlabel("window start (rate percentile)")
print("all values are within the axes (autoscaled)")

By subject, the averaged curves' Spearman correlations.

In [ ]:
subj = np.array([s["subject"] for s in S])
for k, v in CURVES.items():
    print(k, {sb: round(stats.spearmanr(POS, np.nanmean(v[subj == sb], 0)).statistic, 3) for sb in np.unique(subj)})
fastest = CURVES["change"][:, -1]
print("POST minus PRE change in the fastest window per session:", np.round(fastest, 4).tolist(),
      "sign-rank p =", round(stats.wilcoxon(fastest).pvalue, 3))

**Result: reproduced** (fig4c-pre-fast-coactivation). PRE coactivation rises with firing rate, from
0.0208 in the slowest window to 0.0486 in the fastest; the Spearman correlation on the averaged
curve is 0.861 and is positive in 6 of 8 sessions. The change in the fastest window does not
differ from 0 over sessions (sign-rank p = 0.383), consistent with the statement that these
correlations remain into POST.

**Result: not reproduced** (fig4c-slow-coactivation-increase). The POST minus PRE change does not
fall with firing rate (Spearman correlation -0.068 on the averaged curve, negative in 5 of 8
sessions); the criterion needs -0.5 or below.

**Result: not reproduced** (fig4c-slow-replay). The partial correlation of MAZE running and POST
coactivation given PRE is 0.186 in the slowest window and 0.115 in the fastest, but the Spearman
correlation over windows is 0.095 and is negative in only 4 of 8 sessions; the criterion needs
-0.5 or below and 5 sessions.

## rate-split-medians

> Place cells divided into equal groups by sleep firing rate have medians of 0.39 Hz (low-rate) and
> 1.12 Hz (high-rate).

Sleep firing rate is the rate in non-REM sleep of PRE and POST. Reading (a): median split within
each session; reading (b): median split of all place cells pooled. The rate over all of PRE and
POST is a variant.

In [ ]:
def split_medians(col):
    out = {}
    lo_a = CELLS.groupby("session")[col].transform(lambda x: x.rank(method="first") <= len(x) // 2)
    out["a"] = (CELLS.loc[lo_a, col].median(), CELLS.loc[~lo_a, col].median())
    lo_b = CELLS[col].rank(method="first") <= len(CELLS) // 2
    out["b"] = (CELLS.loc[lo_b, col].median(), CELLS.loc[~lo_b, col].median())
    return out, lo_a, lo_b
SPLIT, LOW_A, LOW_B = split_medians("rate_sleep")
for r, (lo, hi) in SPLIT.items():
    print(f"reading ({r}): low median {lo:.3f} Hz, high median {hi:.3f} Hz; within tolerance: "
          f"{abs(lo - 0.39) <= 0.039 and abs(hi - 1.12) <= 0.112}")
v, _, _ = split_medians("rate_prepost")
print("variant, rate over all of PRE and POST:", {k: tuple(round(x, 3) for x in t) for k, t in v.items()})
v, _, _ = split_medians("rate_session")
print("variant, session rate:", {k: tuple(round(x, 3) for x in t) for k, t in v.items()})
print("sleep rate of all place cells: median", round(CELLS.rate_sleep.median(), 3), "range",
      round(CELLS.rate_sleep.min(), 3), "to", round(CELLS.rate_sleep.max(), 3))
print(CELLS.assign(low=LOW_A).groupby(["subject", "low"]).rate_sleep.median().unstack().round(3).to_string())

**Result: partially reproduced.** Split by non-REM sleep rate, the low-rate median is 0.435 Hz
under both readings, just outside 0.39 +/- 0.039, and the high-rate median 1.091 (a) or 1.092 (b),
within 1.12 +/- 0.112. With the rate over all of PRE and POST (a variant) both are within
tolerance (0.395 and 1.041). The split is of 417 place cells, not the paper's 491.

## figs16-low-rate-info-gain

> Low-rate neurons carried more spatial information per spike on the maze, and increased their
> within-ripple firing-rate gain from PRE to POST more than high-rate cells.

In [ ]:
def low_high(col, low):
    A = CELLS.loc[low, col].dropna(); B = CELLS.loc[~low, col].dropna()
    p = stats.mannwhitneyu(A, B).pvalue
    print(f"{col}: low-rate n={len(A)} median {A.median():.3f}; high-rate n={len(B)} median {B.median():.3f}; rank-sum p={p:.3g}")
    return A.median() > B.median() and p < 0.05
print("reading (a), split within each session:")
ok_a = [low_high("info", LOW_A), low_high("gain_change", LOW_A)]
print("reading (b), pooled split:")
ok_b = [low_high("info", LOW_B), low_high("gain_change", LOW_B)]
print("held under (a):", ok_a, " under (b):", ok_b)
print(CELLS.assign(low=LOW_A).groupby(["subject", "low"])[["info", "gain_change"]].median().unstack().round(3).to_string())

**Result: partially reproduced.** Under reading (a), which the criterion selects as neither
reading matched, low-rate cells carry more spatial information (median 0.975 against 0.622 bits
per spike, p = 4.1e-08), but their ripple-gain change from PRE to POST is not larger than that of
high-rate cells (0.251 against 0.155, p = 0.948). Reading (b) gives the same.

## fig4d-slow-sequences

> Slow-firing, but not fast-firing, neurons increased their contribution to sequences from PRE to
> POST sleep, in a decoding restricted to each group.

In [ ]:
print(REPLAY.loc[["base", "low_rate", "high_rate"], ["n_PRE", "n_POST", "mean_PRE", "mean_POST", "diff",
                                                    "p_one_sided", "present", "sessions_POST_gt_PRE", "diff_within", "p_within"]].round(4).to_string())
for name in ("low_rate", "high_rate"):
    print(name, "POST minus PRE mean rZ by subject:", by_subject(name))
print("linear-track sessions only:")
print(REPLAY_LIN.loc[["base", "low_rate", "high_rate"], COLS].round(4).to_string())

**Result: not reproduced.** On the pooled test there is no replay with either half (low-rate
POST minus PRE -0.024, p = 0.964; high-rate -0.049, p = 0.9999), as for all cells. In the
within-session variant both halves show replay, larger for low-rate cells (+0.116 against
+0.058, both p <= 0.0001), which is the partial pattern of the criterion. On the linear-track
sessions alone the pooled test also finds replay in both halves, larger for low-rate cells
(+0.077, p = 0.0046, against +0.044, p = 0.018).

## fig4e-gain-sequences

> PRE to POST increases in sequence content were limited to cells with high ripple-specific
> recruitment (gain).

In [ ]:
print(REPLAY.loc[["base", "high_gain", "low_gain"], ["n_PRE", "n_POST", "mean_PRE", "mean_POST", "diff",
                                                    "p_one_sided", "present", "sessions_POST_gt_PRE", "diff_within", "p_within"]].round(4).to_string())
for name in ("high_gain", "low_gain"):
    print(name, "POST minus PRE mean rZ by subject:", by_subject(name))
print("linear-track sessions only:")
print(REPLAY_LIN.loc[["base", "high_gain", "low_gain"], COLS].round(4).to_string())

**Result: not reproduced.** On the pooled test neither half shows replay (high-gain -0.001,
p = 0.71; low-gain -0.111, p = 1.0). In the within-session variant high-gain cells show replay
(+0.111, p < 0.0001) and low-gain cells do not (+0.016, p = 0.14), the paper's pattern. On the
linear-track sessions alone the pooled test finds replay in neither half (+0.023, p = 0.25;
+0.010, p = 0.21) and the within-session variant finds it in both (+0.045, p = 0.029; +0.050,
p = 0.0028), so the difference between the halves in the within-session variant over all
sessions comes from the circular-track sessions.

## Summary

| Claim | Outcome |
|---|---|
| sessions-and-tracks | reproduced |
| place-cell-count (491) | not reproduced (417; 480 at 0.5 Hz) |
| fig1c-post-gt-pre | not reproduced by the pooled test; higher within all 8 sessions |
| fig1c-maze-gt-pre | reproduced |
| fig2b-strong-contributors (216) | blocked, methods_underspecified |
| fig2-pre-post-overlap (73%) | blocked, methods_underspecified |
| fig2d-shuffle-high-dpcc | partially reproduced |
| fig2e-remove-plastic | blocked, methods_underspecified |
| figs2-track-distribution | blocked, methods_underspecified |
| fig3a-plastic-specificity | blocked, methods_underspecified |
| fig3b-plastic-improve | blocked, methods_underspecified |
| fig4a-rigid-faster | blocked, methods_underspecified |
| fig4a-plastic-ripple-burst-coactivation | blocked, methods_underspecified |
| fig4c-pre-fast-coactivation | reproduced |
| fig4c-slow-coactivation-increase | not reproduced |
| fig4c-slow-replay | not reproduced |
| rate-split-medians | partially reproduced |
| figs16-low-rate-info-gain | partially reproduced |
| fig4d-slow-sequences | not reproduced |
| fig4e-gain-sequences | not reproduced |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

How the analysis differs from the paper's:

- The supplementary Materials and Methods were not available. Place cells, candidate events
  (population bursts holding a ripple on one LFP channel), the decoder's details, the rZ shuffle,
  the PCC and its test, bursting, ripple gain and the field and in-field measures are common
  choices fixed before the analysis.
- The sequence score is decoded with each direction's template on the linear tracks, the event
  taking the larger rZ, and with one template on the circular track. The circular sessions score
  lower in every epoch and supply a larger share of POST than of PRE events, so pooling all
  events, as the criteria fix it, hides the POST over PRE increase that every session shows and
  that the same test finds on each kind of track. The within-session variant recovers the
  paper's pattern for fig2d, fig2e and fig4e, and the linear-track sessions alone for fig2d and
  fig2e; for fig4d both give the criterion's partial pattern. The outcomes are judged on the
  pooled test.
- The PCC test fixed in the criterion, a signed-rank test of a cell's contributions over events,
  has a false-positive rate of 56%, mostly on the linear tracks, where the contribution is taken
  on the template the event was selected for. This blocks the eight claims that rest on the
  selection of strongly contributing cells. A best-template variant of the contribution does
  better (21%) but is still far from 5%.
- dPCC and the rigid and plastic classes are measured on the same events that are decoded again
  for fig2d and fig2e. A cross-validated control (cells chosen on even-numbered events, replay
  measured on odd-numbered ones) shows that shuffling the top 10 to 20% of cells reduces replay
  out of sample without removing it, while removing plastic cells still removes it on the linear
  tracks and within sessions.

What was found about the dandiset:

- The data hold what the paper's replay analyses need: sorted units with a cell type, positions,
  epochs, sleep states and the LFP, all on one clock (shown by decoding position, ripple-burst
  coincidence and the theta/delta ratio of the sleep states). No candidate events, ripples, place
  cell lists or PCC values are stored, so the paper's selections cannot be checked against it.
- In versions 0.210812.1516 and 0.250624.0426 the position series store the sampling period
  (0.0256 s) as their rate; version 0.261005.2101 corrects it.
- In Gatsby_08022013 the POST epoch ends at 30413.6 s while the spikes, LFP and states run to
  32002 s.
- On the circular track the animals ran toward decreasing linearized position; the description
  says the position increases clockwise but not which way they ran.
- The description lists neck EMG and accelerometer signals; the files hold neither.
- Electrode `location` is "unknown" for every channel; the units carry the hemisphere (lCA1 or
  rCA1).